# Assignment 3: Bond futures and cashflow products

**Aditya Amit Shah | as22008**  
FRE-GY 9743 - Special Topics: Models to Markets

This notebook contains the written answers to Questions 1-3, reproducible bond calculations, a conditional CTD scenario example, and the completed computational examples. Run all cells from the repository root. Rates are annual decimal rates; bond prices are per $100 face value.

## Q1. Normalized forward price

**Conventions.** Use the assignment's simple forward-carry model: a regular semiannual bond, valuation just after a coupon payment, exactly 9 years (18 payments) remaining, simple annual repo, and an exact three-month horizon $T=0.25$. Thus $AI_0=0$, the $100 spot price is both clean and dirty, and the next $2 coupon occurs in six months, after delivery. No calendar or repo day-count convention is supplied, so use exact year fractions.

### 1) Three-month forward

At delivery, half of the next semiannual coupon has accrued:

$$AI_T=2\frac{0.25}{0.5}=1.00.$$

The dirty forward is the financed dirty spot price:

$$BF_{\mathrm{dirty}}=100(1+0.02\times0.25)=100.50.$$

Futures are quoted on a clean-price basis, so the forward used in the normalization is

$$\boxed{BF=BF_{\mathrm{dirty}}-AI_T=99.50.}$$

There is no coupon payment during the carry period. Accrual of $1 is not a received coupon and must not be treated as interim cash.

### 2) Conversion factor

Using the question's 9-year bond at the valuation date and nominal 6% yield with semiannual compounding:

$$P_{6\%}=\sum_{j=1}^{18}\frac{2}{(1.03)^j}+\frac{100}{(1.03)^{18}}
=2\frac{1-(1.03)^{-18}}{0.03}+100(1.03)^{-18}
=86.2464869205.$$

Initial accrued interest is zero, so this is also the clean price:

$$\boxed{K=P_{6\%}/100=0.862464869205427.}$$

### 3) Normalized forward

$$\boxed{BF/K=99.50/0.862464869205427=115.3670178957.}$$

Retain full precision in $K$; the displayed futures quote rounds to **115.3670**.

### 4) What the conversion factor does

$K$ expresses each deliverable bond's clean price relative to par under the same 6% yield benchmark. A bond with a 4% coupon is below par at 6%, so $K<1$ and dividing by it scales the bond's clean forward upward. The adjustment accounts for coupon and maturity differences and makes unlike bonds comparable in futures-price units. It does not make their yields, durations or financing costs identical.

Keep accrued interest separate from this clean-price adjustment. Finance the **dirty** spot price, deduct delivery accrued interest to form the **clean** forward, and then divide by $K$. The cash invoice paid to a delivering short is

$$\text{Invoice}_T=FK+AI_T.$$

For example, the unrounded normalized price gives $FK+AI_T=99.50+1.00=100.50$, exactly the financed acquisition cost. Simply dividing the dirty forward by $K$ would incorrectly put accrued interest into the futures quote.

**Conversion-factor timing.** The main answer uses the 9-year spot-date clean price specified above; this choice reproduces the futures quote in Q2. Actual exchange factors follow contract-specific delivery-month and rounding conventions. If the question instead meant the stylized 6% clean price *at delivery*, the factor would be

$$K_T=\frac{P_{6\%}\sqrt{1.03}-1}{100}=0.865306243627,$$

and $BF/K_T=114.9881914442$. This is a different convention, not the factor used in the main answer. For a given futures contract, its assigned factors remain fixed when analyzing scenarios.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

face, coupon_rate, frequency, years_remaining = 100.0, 0.04, 2, 9
spot_clean, repo_rate, delivery_time = 100.0, 0.02, 0.25
coupon = face * coupon_rate / frequency
periods = np.arange(1, frequency * years_remaining + 1)
ai_spot = 0.0
ai_delivery = coupon * delivery_time / (1 / frequency)
spot_dirty = spot_clean + ai_spot
forward_dirty = spot_dirty * (1 + repo_rate * delivery_time)
forward_clean = forward_dirty - ai_delivery
benchmark_clean = (coupon / (1 + 0.06 / frequency)**periods).sum()
benchmark_clean += face / (1 + 0.06 / frequency)**periods[-1]
conversion_factor = benchmark_clean / face
normalized_forward = forward_clean / conversion_factor
futures_quote = 115.3670
implied_repo = ((futures_quote * conversion_factor + ai_delivery) / spot_dirty - 1) / delivery_time

assert np.isclose(ai_delivery, 1.0, rtol=0, atol=1e-12)
assert np.isclose(forward_clean, 99.5, rtol=0, atol=1e-12)
assert np.isclose(conversion_factor, 0.8624648692054272, rtol=0, atol=1e-14)
assert np.isclose(normalized_forward, 115.36701789565932, rtol=0, atol=1e-10)
assert abs(implied_repo - repo_rate) < 1e-6  # Quoted F is rounded to four decimals.
assert np.isclose(normalized_forward * conversion_factor + ai_delivery, forward_dirty)

results = pd.DataFrame({
    "Quantity": ["Delivery accrued interest", "Dirty forward", "Clean forward",
                 "6% clean benchmark price", "Conversion factor", "Normalized forward",
                 "Implied repo (annual decimal)"],
    "Value": [ai_delivery, forward_dirty, forward_clean, benchmark_clean,
              conversion_factor, normalized_forward, implied_repo],
})
display(results.style.format({"Value": "{:.12f}"}))
print(f"Implied repo: {100 * implied_repo:.10f}%")

,Quantity,Value
0,Delivery accrued interest,1.000000000000
1,Dirty forward,100.500000000000
2,Clean forward,99.500000000000
3,6% clean benchmark price,86.246486920543
4,Conversion factor,0.862464869205
5,Normalized forward,115.367017895659
6,Implied repo (annual decimal),0.019999382625


Implied repo: 1.9999382625%


## Q2. Cheapest to deliver and implied repo

### 1) Implied repo for $F=115.3670$

Implied repo is the financing rate that makes buying the bond and delivering it against a short futures position break even. Set the delivery invoice equal to the financed dirty spot cost:

$$FK+AI_T=S_{\mathrm{dirty}}(1+r^{\mathrm{imp}}T).$$

Therefore,

$$\boxed{r^{\mathrm{imp}}=\frac{FK+AI_T-S_{\mathrm{dirty}}}{S_{\mathrm{dirty}}T}
=\frac{115.3670(0.862464869205427)+1-100}{100(0.25)}
=0.0199993826249\approx1.9999382625\%.}$$

It is effectively **2%**, with the small difference caused by rounding the futures quote. Using the alternative delivery-date factor from the convention note would instead give 3.3111416339%.

### 2) Selecting CTD using implied repo and market repo

With common general-collateral funding and otherwise comparable terms, the usual CTD criterion is the **highest implied repo**, since it gives the best return from buying the bond and delivering into a short futures position. Compare this return with the funding rate:

- $r_x^{\mathrm{imp}}>r_{\mathrm{mkt}}$: buying and financing the bond, then delivering against short futures, has positive carry profit before costs. Demand for the bond and selling of futures tend to close the opportunity.
- $r_x^{\mathrm{imp}}<r_{\mathrm{mkt}}$: that bond's implied return is below the financing cost, so delivering it through this trade is unattractive at GC funding.
- In the ideal equilibrium stated in the question, $F=\min_x BF_x/K_x$, the binding CTD has $r_x^{\mathrm{imp}}=r_{\mathrm{mkt}}$ and other bonds have $r_x^{\mathrm{imp}}\leq r_{\mathrm{mkt}}$. The Q1 bond is consistent with being CTD if no other basket member has a lower normalized forward.

The **short futures holder** chooses which eligible bond to deliver; the exchange administers delivery. We cannot identify the actual basket's CTD from this one bond alone.

For exact selection in the assignment's model, calculate

$$x^*=\arg\min_x\frac{BF_x(S_x,r_x^{\mathrm{actual}},T)}{K_x}.$$

Use each bond's actual financing rate if it trades special in repo. A simple unweighted ranking of implied-repo minus actual-repo spreads can be misleading when bonds have different funding exposures. With no interim coupons,

$$\frac{BF_x}{K_x}-F=
\frac{S_{x,\mathrm{dirty}}T}{K_x}(r_x^{\mathrm{actual}}-r_x^{\mathrm{imp}}).$$

The coefficient differs by bond. Similarly, dollar carry profit is $S_{x,\mathrm{dirty}}T(r_x^{\mathrm{imp}}-r_x^{\mathrm{actual}})$, so a return ranking and an equal-face dollar-profit ranking need not coincide away from the ideal equilibrium.

If coupons $C_i$ are received before or on delivery and reinvested at the same simple repo rate, the clean forward is

$$BF_x=S_{x,\mathrm{dirty}}(1+r_xT)
-\sum_{0<t_i\leq T}C_i[1+r_x(T-t_i)]-AI_x(T).$$

Use that full cashflow carry equation rather than the no-coupon formula when appropriate.


## Q3. Scenario analysis and why CTD matters

A future CTD can be predicted **conditionally on a market scenario**, not known with certainty. Advance the valuation date by $N$ days and retain the same delivery date and contract conversion factors. Then, for every eligible bond:

1. Rebuild the remaining cashflows and accrued interest; account for coupons received and the shorter time to delivery.
2. Reprice the bond under the assumed yield curve, including parallel moves, steepening/flattening and bond-specific relative-value changes.
3. Apply the assumed financing rate, including any repo specialness, to calculate its clean forward.
4. Divide by its fixed $K_x$ and select the minimum. Repeat across scenarios to locate CTD-switch boundaries.

A yield rise generally favors a longer-duration, lower-coupon bond because its price falls more; a rally generally favors a shorter-duration, higher-coupon bond. The 6% conversion benchmark influences these tendencies, but curve shape, repo and relative value can overturn them.

**Illustrative 30-day scenario.** The question supplies no complete basket, so the next cell uses two hypothetical eligible bonds: A has a 4% coupon and 9 years remaining; B has a 6% coupon and 8 years remaining. Both have $100 face, regular semiannual payments and a coupon date at the original valuation date. Observe them 30/360 years later, with the original delivery still at $T=0.25$ years, and assume common 2% repo. No coupon occurs before delivery. The assumed flat yield applies to both bonds at the scenario date; their conversion factors are fixed at the original 6% benchmark.

If $P_x(y)$ is the original coupon-date dirty value under scenario yield $y$, the dirty value at observation time $s$ is $P_x(y)(1+y/2)^{2s}$. Financing from $s$ to delivery and deducting $AI_x(T)=100c_xT$ gives

$$N_x(s;y)=\frac{P_x(y)(1+y/2)^{2s}[1+r(T-s)]-100c_xT}{K_x}.$$

These are possible future valuations, not observed market data. The example shows B cheapest at 3% and 5%, but A cheapest at 7% and 9%.

CTD determines the delivery invoice, funded acquisition cost and profit from the delivery trade. It also drives futures rate risk and hedge ratios. Per the same face amount,

$$DV01_{\mathrm{futures}}\approx\frac{DV01_{\mathrm{CTD}}}{K_{\mathrm{CTD}}}.$$

A switch changes this exposure and can leave a hedge designed around today's CTD less effective. Traders therefore monitor delivery cost, repo exposure and the value of the short's delivery choice, rather than assuming today's CTD will remain cheapest.


In [2]:
basket = [("A: 4%, 9 years", 0.04, 9), ("B: 6%, 8 years", 0.06, 8)]
observation_time = 30 / 360
remaining_carry = delivery_time - observation_time
scenario_yields = [0.03, 0.05, 0.06, 0.07, 0.09]

def coupon_date_price(rate, years, yield_rate):
    payment_numbers = np.arange(1, 2 * years + 1)
    discounts = (1 + yield_rate / 2)**(-payment_numbers)
    return 100 * rate / 2 * discounts.sum() + 100 * discounts[-1]

fixed_factors = {label: coupon_date_price(rate, years, 0.06) / 100
                 for label, rate, years in basket}
scenario_rows = []
for yield_rate in scenario_yields:
    values = {}
    for label, rate, years in basket:
        dirty_at_observation = coupon_date_price(rate, years, yield_rate) * (1 + yield_rate / 2)**(2 * observation_time)
        dirty_at_delivery = dirty_at_observation * (1 + repo_rate * remaining_carry)
        clean_forward_at_observation = dirty_at_delivery - 100 * rate * delivery_time
        values[label] = clean_forward_at_observation / fixed_factors[label]
    scenario_rows.append({"Scenario yield": f"{100 * yield_rate:.0f}%", **values,
                          "CTD": min(values, key=values.get)})
scenario_table = pd.DataFrame(scenario_rows)
assert scenario_table.iloc[0]["CTD"] == "B: 6%, 8 years"
assert scenario_table.iloc[-1]["CTD"] == "A: 4%, 9 years"
assert remaining_carry > 0
display(scenario_table.style.format({label: "{:.6f}" for label, _, _ in basket}))

,Scenario yield,"A: 4%, 9 years","B: 6%, 8 years",CTD
0,3%,124.601654,120.403005,"B: 6%, 8 years"
1,5%,107.270225,105.823368,"B: 6%, 8 years"
2,6%,99.669374,99.328842,"B: 6%, 8 years"
3,7%,92.694375,93.308152,"A: 4%, 9 years"
4,9%,80.404669,82.540418,"A: 4%, 9 years"


### References and model scope

The numerical answers above follow the assignment's simplified carry equation and explicitly stated conventions. Exchange-specific delivery options, daily futures settlement, transaction costs and bid/ask spreads are outside this calculation.

- [CME: conversion factor, clean price and delivery invoice](https://www.cmegroup.com/trading/interest-rates/us-treasury-futures-conversion-factor-lookup-tables.html).
- [CME: actual delivery-month conversion-factor conventions](https://www.cmegroup.com/articles/2024/calculating-us-treasury-futures-conversion-factors.html).
- [CME: implied repo and CTD](https://www.cmegroup.com/education/courses/introduction-to-treasuries/get-to-know-treasuries-ctd).
- [CME: Understanding Treasury Futures, including CTD and hedging](https://www.cmegroup.com/education/files/understanding-treasury-futures.pdf).

## Computational part: implementation explanation

`ProductFixedAccruedCashflow` initializes the shared signed notional, currency and payment fields through `ProductCashflow`, stores its accrual dates and conventions, and computes a **year fraction** with the library's `accrued()` function. It has no fixed-rate field, so neither the year fraction nor the display is an interest payment or present value.

`ProductOvernightIndexCashflow` obtains its index from `IndexRegistry`, resolves an explicit end date or tenor with the index's calendar and business-day convention, derives currency from that index, and stores the compounding method and annual decimal spread. Pricing would additionally need overnight fixings or a forecast curve; no such inputs are required for contract creation.

Both products implement `accept(visitor)` by returning `visitor.visit(self)`. `ProductDisplayVisitor.visit` uses `singledispatchmethod` to choose the handler for the concrete product type. Each handler resets the common rows and adds its own contract fields, so reusing a visitor does not mix two products' attributes. This puts the display operation in the visitor while keeping the product classes focused on contract data.

The supplied examples below exercise the public creation/display APIs. Additional verification after the examples checks serialization, business-day handling and the numerical/contract assertions through a separate test suite.


# HW3: Atomic cashflows and the Visitor pattern

This assignment develops two skills:

1. **Model cashflow contracts.** Implement `ProductFixedAccruedCashflow` and `ProductOvernightIndexCashflow` using the provided `Product` and `ProductCashflow` bases. Understand their contractual fields, including dates, notional, currency, accrual conventions, index and spread. Each object is a building block for a larger product, such as a swap.
2. **Use the Visitor pattern.** Collect and display each product's attributes through a visitor. This demonstrates how to add an operation without changing the product classes.

Read [IV. Visitor Pattern](https://immense-chocolate-92e.notion.site/IV-Visitor-Pattern-2d4831d0b1d58191bc36ebae3d771d58); implement `accept()` and the display visitor to produce the tables below.

The six `#TODO` blocks are completed in `fixedincomelib/product/linear_products.py` and `fixedincomelib/product/product_display_visitor.py` for the examples below.

Use the provided `qfCreateProductxxx` APIs to create product objects and `qfDisplayProduct` to inspect their fields.

These objects also form a foundation for a later `value_engine`, which can combine their contract attributes with the necessary rates and market data to calculate cash amounts and present values. The engine reads the product objects directly; the display tables are for inspection.

In [3]:
import sys
import QuantLib as ql

from fixedincomelib import (
    qfCreateProductFixedAccruedCashflow,
    qfCreateProductOvernightIndexCashflow,
    qfDisplayProduct,
)
from fixedincomelib.product import (
    ProductCashflow,
    ProductFixedAccruedCashflow, ProductOvernightIndexCashflow,
)

print(f"Python {sys.version.split()[0]} | QuantLib {ql.__version__}")

Python 3.12.14 | QuantLib 1.41


## 1. Fixed-accrual cashflow

The API returns a `ProductFixedAccruedCashflow` object containing contract fields and an accrual year fraction. `qfDisplayProduct` displays its fields; the fraction is checked below. This class has no fixed-rate field.

In [4]:
fixed = qfCreateProductFixedAccruedCashflow(
    effective_date="2026-01-02",
    termination_date="2026-04-02",
    currency="USD",
    notional=1_000_000,
    accrual_basis="ACT/360",
    payment_date="2026-04-06",
    business_day_convention="F",
    holiday_convention="USGS",
)
assert isinstance(fixed, ProductCashflow)
assert isinstance(fixed, ProductFixedAccruedCashflow)
print("Created:", type(fixed).__name__)
qfDisplayProduct(fixed)

Created: ProductFixedAccruedCashflow


,Name,Value
0,Product Type,PRODUCT_FIXED_ACCRUED
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,Accrual Basis,ACT/360
7,Payment Date,2026-04-06
8,Business Day Convention,F
9,Holiday Convention,USGS


In [5]:
# 90 calendar days / 360. This number is a year fraction, not a cash amount.
assert fixed.accrued == 90 / 360
assert fixed.termination_date.ISO() == "2026-04-02"
assert fixed.payment_date.ISO() == "2026-04-06"
assert fixed.first_date == fixed.effective_date
assert fixed.last_date == fixed.payment_date
print(f"Accrual year fraction: {fixed.accrued:.6f}")
print("Contractual accrual end:", fixed.termination_date.ISO())
print("Payment date:", fixed.payment_date.ISO())

Accrual year fraction: 0.250000
Contractual accrual end: 2026-04-02
Payment date: 2026-04-06


## 2. Overnight-index cashflow

The API returns a `ProductOvernightIndexCashflow` object. It resolves the tenor to a termination date, obtains currency from the index, and stores the contract fields.

`0.001` is 10 basis points as an annual decimal rate. Negative notional implies `SHORT`. Creation and display do not calculate an interest amount.

In [6]:
overnight = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert isinstance(overnight, ProductCashflow)
assert isinstance(overnight, ProductOvernightIndexCashflow)
assert overnight.notional == -1_000_000
assert overnight.currency.value_str == "USD"
print("Created:", type(overnight).__name__)
qfDisplayProduct(overnight)

Created: ProductOvernightIndexCashflow


,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,-1000000
2,Currency,USD
3,Long Or Short,SHORT
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SOFR-1B
7,Compounding Method,COMPOUND
8,Spread,0.001
9,Payment Date,2026-04-06


In [7]:
# Date and tenor inputs resolve to the same termination date.
overnight_by_date = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="2026-04-02",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert overnight.termination_date == overnight_by_date.termination_date
assert overnight.payment_date == overnight_by_date.payment_date
print("Date and tenor inputs agree:", overnight.termination_date.ISO())

Date and tenor inputs agree: 2026-04-02


## 3. Currency and payment defaults

Supported indices are `SOFR-1B`, `FF-1B` and `SONIA-1B` (case-insensitive). Currency comes from the index, and an omitted payment date defaults to termination. Compounding labels are `simple`, `arithmetic` and `compound`.

In [8]:
sonia = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SONIA-1B",
    notional=500_000,
    compounding_method="arithmetic",
)
assert sonia.currency.value_str == "GBP"
assert sonia.payment_date == sonia.termination_date
qfDisplayProduct(sonia)

,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,500000
2,Currency,GBP
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SONIA-1B
7,Compounding Method,ARITHMETIC
8,Spread,0.0
9,Payment Date,2026-04-02


## Verification

The checks below verify serialization round trips and run the independent product tests. These cover actual cashflow behavior: default and explicit payments, calendar-adjusted termination dates, signed notionals, supported indices and methods, invalid inputs, visitor return values and display-row isolation.


In [9]:
import unittest
from pathlib import Path
from fixedincomelib.product import ProductDisplayVisitor

for product in (fixed, overnight, overnight_by_date, sonia):
    restored = type(product).deserialize(product.serialize())
    assert restored.serialize() == product.serialize()
    assert qfDisplayProduct(restored).equals(qfDisplayProduct(product))

visitor = ProductDisplayVisitor()
fixed.accept(visitor)
assert len(visitor.display()) == 10
overnight.accept(visitor)
assert len(visitor.display()) == 10  # Common rows are replaced for each visit.
assert "ON Index" in set(visitor.display()["Name"])
assert "Accrual Basis" not in set(visitor.display()["Name"])

suite = unittest.defaultTestLoader.discover(str(Path("tests")), pattern="test_*.py")
assert suite.countTestCases() > 0
run_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert run_result.wasSuccessful(), "Product verification failed"
print(f"All numerical assertions and {run_result.testsRun} product tests passed.")

test_accept_returns_the_visitor_result (test_products.DisplayVisitorTests.test_accept_returns_the_visitor_result) ... 

ok


test_display_rejects_nonproducts (test_products.DisplayVisitorTests.test_display_rejects_nonproducts) ... 

ok


test_fixed_display_order_and_types (test_products.DisplayVisitorTests.test_fixed_display_order_and_types) ... 

...

ok


test_zero_length_dates_and_optional_payment_representations (test_products.OvernightCashflowTests.test_zero_length_dates_and_optional_payment_representations) ... 

ok


----------------------------------------------------------------------
Ran 20 tests in 0.007s

OK


All numerical assertions and 20 product tests passed.
